# Curry College SMIF – Analytics Notebook

Work through the modules **in order**. Every grey box is a *cell*: click its ▶ **Play** button (or press **Shift+Enter**) and wait for the ✓ before moving on.

| Module | What you do | Where you type |
|---|---|---|
| **0** | Set up, choose dates, download data | the ✏️ Settings cell |
| **1** | Our six ETFs: what we bought, weights, performance since 10/19 | `data/trades.csv` |
| **2** | Top-10 stocks vs the rest since 10/19 | the SMIF Portfolio Excel file |
| **3** | Rebuild the Excel Sharpe optimization: stats, CAPM, min-variance, max-Sharpe, efficient frontier | dates in Settings |
| **4** | Repeat with the seven new ETFs | `sm.CANDIDATES` in Settings |
| **5** | Top holdings, overlap, ETF vs its top N, add your own ETF | `data/holdings.csv` (via `sm.add_holdings`) |
| **6** | Event studies: midterms, Fed hikes, oil spikes | event dates |
| **7** | Extra: factor exposure | nothing |

**Look for the ✏️ symbol: that is where you may type. Everything else just press Play.**
*Nothing here is investment advice. It is a learning tool and one input into our decisions.*

---
# Module 0 – Setup

### 0a. Install the tools (once per Codespace)
Python needs add-on packages: **pandas** (tables), **numpy** (math), **scipy** (the optimizer, like Excel Solver), **matplotlib** (charts), **yfinance** (prices from Yahoo Finance), **openpyxl** (reads Excel).

In [ ]:
%pip install -q numpy pandas scipy matplotlib yfinance openpyxl

### 0b. ✏️ Settings (the only place you need to type in Module 0)
| Setting | Meaning |
|---|---|
| `TRADE_DATE` | the day we bought the six ETFs |
| `START`, `END` | the **date window** the models use, like the start/end dates in our Excel file. `END = None` means "up to the latest data" |
| `MAX_W` | the optimizer may not put more than this share (0.35 = 35%) in one ETF |
| `ADD_SIZE` | in the candidate test we pretend to move this share of the portfolio into a new ETF |
| `CANDIDATES` | the new ETFs we are researching (ticker: label) |

*Changed a date? Run this cell again, then run `px = sm.load()` (Module 0d) and the cells you want to redo.*

In [ ]:
import smif_models as sm

sm.TRADE_DATE = "2026-10-19"
sm.START = "2021-07-01"     # ✏️ start of the modeling window (YYYY-MM-DD)
sm.END = None               # ✏️ end of the window, e.g. "2026-10-26". None = today
sm.MAX_W = 0.35             # ✏️ max weight per ETF in the optimizer
sm.ADD_SIZE = 0.10          # ✏️ share moved into a candidate in the screen
sm.CANDIDATES = {"XLU": "Utilities", "IBB": "Biotech", "PAVE": "Infrastructure",
                 "XLP": "Consumer Staples", "SMH": "Semiconductors",
                 "IJH": "Mid-cap", "IYG": "Financial Services"}

### 0c. Download prices (needs internet, 1–2 minutes)
Downloads daily prices back to the year 2000 for all ETFs (adjusted for dividends and splits, so they are **total returns**) and since 2023 for the individual stocks, plus the 3-month Treasury bill rate (our risk-free rate). Saved in `data/prices.csv`. **Run again whenever you want fresher prices.** You do *not* need to re-run it just to change dates.

In [ ]:
sm.update()

### 0d. Load the data
* `px_all` = everything downloaded (used for "since the trade" and event studies).
* `px` = only the dates between your `START` and `END` (used for the models).

It prints the first and last date used, so you can check your window.

In [ ]:
px_all = sm.load(full=True)
px = sm.load()
print("Modeling window:", px.index[0].date(), "to", px.index[-1].date(), "-", len(px), "trading days")
print("Latest price date downloaded:", px_all.index[-1].date())

---
# Module 1 – Our portfolio: what we bought and how it has done since 10/19

### 1a. What we bought
This table comes from **`data/trades.csv`**: the ETF, number of shares, and the **price we paid** on 10/19.
✏️ After a future trade, edit that file (open it in the file list on the left) and update `sm.TRADE_DATE`.

In [ ]:
trades = sm.load_trades()
trades["cost"] = trades.shares * trades.price_paid
trades["weight_at_purchase_%"] = (trades.cost / trades.cost.sum() * 100).round(2)
trades

### 1b. How it has performed from our fill prices to the latest close
The four charts show: the value of the portfolio against what we paid, each ETF's return since we bought it, how the **weights drift** as prices move, and each ETF's **contribution in basis points** (1 bp = 0.01% of the whole portfolio).
**Think about:** which ETF cost us the most, and is that because it fell a lot or because we held a lot of it?

In [ ]:
tab = sm.portfolio_since_trade(px_all)

### 1c. Right now (live price)
Asks Yahoo for each ETF's latest traded price (may be ~15 minutes delayed). Good for a quick check before class or a meeting.

In [ ]:
live = sm.snapshot()

---
# Module 2 – Top-10 stocks vs everything else (since 10/19)

Our ETFs hold hundreds of stocks. Your **SMIF Portfolio Excel file** lists the **top 10 stocks of each ETF** (about 60 stocks, combined across ETFs) with the shares they represent and their 10/19 prices. We compare that top-10 basket with *everything else* in the six ETFs.

Because we use the holdings we actually had on 10/19 over a short period since then, there is no look-ahead problem here.
**Think about:** did the biggest companies help or hurt us? Which single stocks mattered most?

In [ ]:
cum, contrib = sm.top10_vs_rest(px_all)

---
# Module 3 – Building the optimal portfolio (the Excel steps, in Python)

In Excel we (1) took daily returns, (2) found each ETF's average return, volatility and Sharpe ratio, (3) found beta and the CAPM expected return, (4) computed **alpha = actual – CAPM**, (5) built the covariance matrix and (6) used Solver to find weights. Here is the same chain.

✏️ **Change the dates** in the Settings cell (0b), run it, run `px = sm.load()` (0d), then rerun this module. To match the Excel file exactly use `sm.START = "2021-07-01"` and `sm.END = "2026-10-26"`.

### 3a. Return, risk, Sharpe, beta, CAPM and alpha
`alpha_tstat` above 2 (or below −2) means the alpha is unlikely to be luck. In the SMIF data almost none are.

In [ ]:
stats6 = sm.stats(px)

### 3b. Do our ETFs move together? (correlation)
Diversification only helps when correlations are low. The color scale runs 0 to 1 unless a pair is negative.

In [ ]:
corr6 = sm.corr_heatmap(px, sm.universe())

### 3c. Maximum-Sharpe, minimum-variance and the efficient frontier
* **max_sharpe**: the most return per unit of risk (what Solver gave us).
* **min_variance**: the lowest possible risk (the lecture on minimizing standard deviation).
* **risk_parity**: every ETF contributes equal *risk*.
* **CHOSEN**: the weights we actually bought, after the optimization **and** the students' sector research.

**How to read the frontier chart:** each dot is an ETF, the cloud is thousands of random portfolios, the red curve is the best return for each risk level (the **efficient frontier**), the dashed line is the **capital allocation line** from the risk-free rate through the max-Sharpe portfolio. Where did our ✕ land compared with the curve?

In [ ]:
res6 = sm.optimize(px, sm.universe(), label='Current six ETFs')

### 3d. Reflection: what we chose vs what the models said
The chosen weights differ from max-Sharpe on purpose: students added sector research, valuation and common sense. The second chart applies each set of weights from the 10/19 close to today. **It is only a few weeks, so it proves nothing**, but it is a good conversation starter.

In [ ]:
refl6 = sm.reflect(px, px_all)

### 3e. The honest test: out of sample
The optimizer chooses weights using past data, so its in-sample results always look good. This test picks weights using only the previous 2 years, holds them for 3 months, and repeats. **Which method does best on data it has not seen?** (Often the "boring" ones: minimum variance and equal weight.)

In [ ]:
wf6 = sm.walk_forward(px, sm.universe())

---
# Module 4 – Now include the new ETFs

The seven candidates (`sm.CANDIDATES`): **XLU, IBB, PAVE, XLP, SMH, IJH, IYG**. ✏️ To research a different ETF, add it to `sm.CANDIDATES` in Settings (0b), add its holdings in Module 5g, run `sm.update()`, then rerun this module.

**Window warning:** the optimizer needs every ETF to have data for every day, so the shortest history sets the window. PAVE only started in 2017; the notebook prints which ETF limits the window.

### 4a. Risk, return, CAPM and alpha for all 13

In [ ]:
stats13 = sm.stats(px, sm.universe(True))

### 4b. Optimal weights with 13 ETFs
Notice how many ETFs the optimizer ignores (weight 0%). Optimizers love corners; that is a reason not to trust them blindly.

In [ ]:
res13 = sm.optimize(px, sm.universe(True), label='Six ETFs + candidates')

### 4c. Does adding the candidates improve the frontier?
Both curves use exactly the same dates. If the orange curve is not clearly up and to the left of the dark one, the new ETFs did not add much *historically*.

In [ ]:
cmp = sm.compare_frontiers(px)

### 4d. Which single candidate helps most? (screen)
We pretend to move `ADD_SIZE` (10%) of the portfolio into each candidate. **Trust `d_vol` (change in risk) and the correlations most**; the return columns depend on noisy past averages. A negative `d_vol` means adding it would lower risk.

In [ ]:
screen = sm.candidates(px)

### 4e. Out-of-sample test with all 13

In [ ]:
wf13 = sm.walk_forward(px, sm.universe(True))

---
# Module 5 – Top holdings and overlap

Weights inside each ETF come from **`data/holdings.csv`**, built from Morningstar's Stock Intersection and the fund pages. Every row has the ETF, the stock, its weight **inside that ETF**, and an as-of date. A `REST` row is 100% minus the stocks we list.

### 5a. The top 5 of every ETF

In [ ]:
for e in sm.universe(True):
    print(e); print(sm.top_n(e, 5).round(2).to_string(index=False)); print()

### 5b. Which ETFs are really the same thing? (overlap)
Each cell = the % of the ETF that is the same stocks as the other ETF (counting only holdings we list, so it is a minimum). The diagonal shows how much of each ETF our list covers.

In [ ]:
ov = sm.overlap_matrix()

### 5c. Our real stock exposure (look-through)
Adds up each stock across all the ETFs we hold. ✏️ To test a **proposed** portfolio, pass weights as fractions, e.g. `sm.look_through({"XLE": .15, "VHT": .25, "VOO": .30, "XLU": .10, "SMH": .20})`.

In [ ]:
lt = sm.look_through()

### 5d. One ETF vs its top 5 (or 10) stocks
**How well do the biggest holdings explain the whole ETF?** Compare R-squared, tracking error, and the implied return of everything else. ✏️ Change the ETF and `n`.

> ⚠️ **Time caution:** this uses *today's* top holdings on *past* prices, so those stocks are there because they did well (look-ahead bias). Keep windows short (the last year by default) and use it to see **concentration**, not to predict returns.

In [ ]:
etf_stats = sm.etf_vs_topn(px_all, "VHT", n=5)       # ✏️ try "XLE", "SMH", "VOO" ... and n=10

### 5e. Every ETF at once: do more concentrated funds track their top holdings better?

In [ ]:
all_stats = sm.etf_vs_topn_all(px_all, n=5)

### 5f. The whole portfolio vs the top-5 stocks of each of our ETFs
Builds one basket of every stock that is a top-5 holding in any of our six ETFs (weighted by our look-through exposure) and compares it with our actual portfolio. **How much of our portfolio is really a handful of companies?**

In [ ]:
overall_stats, overall_expo = sm.overall_vs_topn(px_all, n=5)

### 5g. ✏️ Found a new ETF? Add its top holdings
1. Open the fund's website (or Morningstar) → **Holdings**. Copy the **top 5–10 stock tickers and their weights (%)** and the **as-of date** shown.
2. Fill in the cell below and run it. Weights are the % **inside the ETF** and must add up to 100 or less.
3. Add the ETF to `sm.CANDIDATES` in Settings, run `sm.update()`, and rerun Modules 4 and 5.

Optional: `sm.yahoo_top_holdings("ITA")` asks Yahoo for today's top 10. **Check it against the fund's website** before saving.

In [ ]:
# EXAMPLE ONLY: weights are made up. Replace with the real numbers from the fund's website.
# sm.add_holdings("ITA",
#                 {"GE": 20.0, "RTX": 18.5, "BA": 8.0, "LMT": 5.0, "NOC": 4.5},
#                 as_of="2026-11-09",
#                 names={"GE": "GE Aerospace", "RTX": "RTX Corp"})

---
# Module 6 – Event studies

**Question:** do certain events move some sectors more than the overall market?

**Method (market model):** for each event we use the ~120 trading days before it to learn how each ETF normally moves with the overall market (VTI): `ETF return = a + b × market return`. Then, around the event, **abnormal return (AR) = actual return − predicted return**. Adding up ARs gives the **cumulative abnormal return (CAR)**. `t` values above 2 on a single event are notable.

**Default events** (day 0 = first trading day on or after the date):
* **Midterm elections**, the day after: 2006, 2010, 2014, 2018, 2022 and 2026 (the 2026 window is still incomplete and marked with *).
* **Fed rate hikes:** 3/16/2022 (first of the cycle), 6/15/2022, 7/26/2023 (last of that cycle).
* **Oil price spikes:** 7/11/2008 (record high), 2/24/2022 (Russia invades Ukraine), 3/8/2022 (post-invasion peak).

✏️ **Check the dates and add your own** in the cell below. Candidate ETFs with no history at the time (like PAVE in 2006) are skipped automatically.
Daily data cannot separate events that happen within the same day; for an announcement after the close, use the next trading day.

In [ ]:
import copy
groups = copy.deepcopy(sm.EVENT_GROUPS)
# ✏️ Add or change events, e.g.:
# groups["Our own event"] = ["2026-11-04"]
# groups["Trump post"] = ["YYYY-MM-DD"]      # the date the MBA assistant is studying
events = sm.event_study(px_all, groups)

**Look at the output:** the left chart shows each ETF's average abnormal return around the event; the right heatmap shows each individual event. **Think about:** energy around oil spikes? Rate-sensitive sectors (utilities, financials) around Fed hikes? Is the pattern consistent across events, or is it one big outlier?

To look at one event in detail, pass `groups={"My event": ["YYYY-MM-DD"]}` to `sm.event_study(px_all, ...)`. To see the per-event numbers, show `events` as a table:

In [ ]:
events[events.group == 'Fed rate hikes'].round(2)

---
# Module 7 (extra) – Factor exposure
A *factor* is something that drives returns, like company size or style. We build factors from ETF spreads (small vs large, value vs growth, momentum, quality, low volatility) and ask what each ETF is really exposed to.

In [ ]:
fac = sm.factors(px)

---
# Wrap-up: your recommendation
For your assigned ETF (and your candidate), write a one-page recommendation: **buy more, hold, reduce or sell**, and for the candidate **add or skip**. Use at least three numbers from this notebook (for example Sharpe and alpha t-stat from 3a, change in volatility from 4d, overlap from 5b, how concentrated it is from 5d) **and** your own sector research (earnings, valuation, economic outlook). Explain what would change your mind.